In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

In [15]:
blocks = ['ac97_top', 'i2c_master_top', 'mc_top', 'sasc_top', 'simple_spi_top', 'spi_top', 'usb_phy', 'aes_cipher_top', 'pcm_slv_top', 'sha256']



df = pd.read_csv('../dataset/i2c_master_top/features/net_features_place.csv', sep=',')
df_y = pd.read_csv('../dataset/i2c_master_top/route_label/net_labels.csv', sep=',')
df_all = df.merge(df_y, on='net_name', how='inner')

non_feature_cols = [
    'net_name',
    'net_dbid',
    'odb_net',
    'status',
    'label'
]

x = df_all.drop(columns=[c for c in non_feature_cols if c in df_all.columns])
y = df_all['label']

for c in x.columns:
    x[c] = pd.to_numeric(x[c], errors='coerce')

x = x.dropna(axis=1, how='all')

nunique = x.nunique()
const_cols = nunique[nunique <= 1].index.tolist()

print(f"Drop constant columns ({len(const_cols)}):")
print(const_cols)

x = x.drop(columns=const_cols)

Drop constant columns (7):
['driver_is_clkbuf', 'has_multiple_drivers', 'is_clock_like', 'is_reg2reg_candidate', 'num_clkbuf_drivers', 'num_drivers', 'num_inout']


In [16]:
x

,bbox_area,bbox_aspect,bbox_cx,bbox_cy,bbox_dx,bbox_dy,bbox_perimeter,bbox_xmax,bbox_xmin,bbox_ymax,...,sink_min_y,sink_quadrant_q1,sink_quadrant_q2,sink_quadrant_q3,sink_quadrant_q4,sink_x_mean,sink_x_std,sink_y_mean,sink_y_std,term_count
0,8.903100,1.023051,44.6920,87.1630,3.018,2.950,11.936,46.201,43.183,88.638,...,88.40,1,0,0,0,49.22,NaN,88.40,NaN,2
1,4.460072,0.515296,42.3590,70.8470,1.516,2.942,8.916,43.117,41.601,72.318,...,72.08,1,0,0,0,44.62,NaN,72.08,NaN,2
2,22.711745,0.667981,50.7505,118.3625,3.895,5.831,19.452,52.698,48.803,121.278,...,121.04,0,1,0,0,49.68,NaN,121.04,NaN,2
3,1.148774,0.035520,40.7820,64.0345,0.202,5.687,11.778,40.883,40.681,66.878,...,66.64,1,0,0,0,43.70,NaN,66.64,NaN,2
4,10.857132,1.470740,7.7180,114.2465,3.996,2.717,13.426,9.716,5.720,115.605,...,115.60,1,0,0,0,8.51,NaN,115.60,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1252,0.045524,0.007930,8.0855,190.4280,0.019,2.396,4.830,8.095,8.076,191.626,...,189.04,0,0,0,1,8.51,NaN,189.04,NaN,2
1253,0.027303,8.403509,9.2355,156.5615,0.479,0.057,1.072,9.475,8.996,156.590,...,156.40,0,0,0,1,9.89,NaN,156.40,NaN,2
1254,0.073248,23.357142,8.7490,153.5180,1.308,0.056,2.728,9.403,8.095,153.546,...,153.68,0,1,0,0,8.51,NaN,153.68,NaN,2
1255,0.026768,8.535714,15.1440,158.9580,0.478,0.056,1.068,15.383,14.905,158.986,...,159.12,0,1,0,0,14.49,NaN,159.12,NaN,2


In [18]:
import os
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# -----------------------------
# Config
# -----------------------------
blocks = ['ac97_top', 'i2c_master_top', 'mc_top', 'sasc_top', 'simple_spi_top', 'spi_top', 'usb_phy']

# choose 2 test blocks
test_blocks = ['spi_top', 'usb_phy']
train_blocks = [b for b in blocks if b not in test_blocks]

base_dir = "../dataset"   # adjust if needed

non_feature_cols = ['net_name', 'net_dbid', 'odb_net', 'status', 'label']

# -----------------------------
# Utils
# -----------------------------
def load_one_block(block: str) -> pd.DataFrame:
    f_feat = os.path.join(base_dir, block, "features", "net_features_place.csv")
    f_lbl  = os.path.join(base_dir, block, "route_label", "net_labels.csv")

    df = pd.read_csv(f_feat, sep=",")
    df_y = pd.read_csv(f_lbl, sep=",")

    # normalize names (important)
    df['net_name'] = df['net_name'].astype(str).str.strip().str.replace('"', '', regex=False)
    df_y['net_name'] = df_y['net_name'].astype(str).str.strip().str.replace('"', '', regex=False)

    df_all = df.merge(df_y, on='net_name', how='inner')
    df_all['block'] = block
    return df_all

def to_numeric_features(df_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    x = df_all.drop(columns=[c for c in non_feature_cols if c in df_all.columns], errors='ignore')
    y = df_all['label'].astype(int)

    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors='coerce')

    # drop columns fully NaN in this block
    x = x.dropna(axis=1, how='all')
    return x, y

def align_feature_columns(X_list: list[pd.DataFrame]) -> list[pd.DataFrame]:
    # union of columns across all blocks, reindex each to the union
    all_cols = sorted(set().union(*[set(X.columns) for X in X_list]))
    return [X.reindex(columns=all_cols) for X in X_list]

def drop_constant_columns(X: pd.DataFrame) -> pd.DataFrame:
    nunique = X.nunique(dropna=True)
    const_cols = nunique[nunique <= 1].index.tolist()
    if const_cols:
        print(f"Drop constant columns ({len(const_cols)}): {const_cols[:20]}{'...' if len(const_cols)>20 else ''}")
        X = X.drop(columns=const_cols)
    else:
        print("Drop constant columns (0)")
    return X

def recall_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    if y_true.sum() == 0:
        return np.nan
    k = int(max(1, min(k, len(y_true))))
    idx = np.argsort(y_score)[::-1][:k]
    return float(y_true[idx].sum() / y_true.sum())

def eval_block(y_true, proba, block_name: str):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)

    # AUCs can fail if only one class in y_true
    try:
        roc = roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        roc = np.nan
    try:
        pr = average_precision_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        pr = np.nan

    k = int(y_true.sum()) if y_true.sum() > 0 else 1
    r_at_k = recall_at_k(y_true, proba, k)

    return {
        "block": block_name,
        "n": len(y_true),
        "pos": int(y_true.sum()),
        "pos_rate": float(y_true.mean()),
        "roc_auc": roc,
        "pr_auc": pr,
        "recall@K(pos)": r_at_k,
        "K": k,
    }

# -----------------------------
# Load data
# -----------------------------
train_dfs = [load_one_block(b) for b in train_blocks]
test_dfs  = [load_one_block(b) for b in test_blocks]

# per-block X,y
train_Xs, train_ys = zip(*[to_numeric_features(df) for df in train_dfs])
test_Xs,  test_ys  = zip(*[to_numeric_features(df) for df in test_dfs])

# align columns across all blocks (train+test)
all_Xs = list(train_Xs) + list(test_Xs)
all_Xs_aligned = align_feature_columns(all_Xs)

train_Xs_aligned = all_Xs_aligned[:len(train_Xs)]
test_Xs_aligned  = all_Xs_aligned[len(train_Xs):]

# concatenate train
X_train = pd.concat(train_Xs_aligned, axis=0, ignore_index=True)
y_train = pd.concat([pd.Series(y) for y in train_ys], axis=0, ignore_index=True).astype(int)

# concatenate test (but keep per-block copies for per-block eval)
X_test_blocks = test_Xs_aligned
y_test_blocks = [pd.Series(y).astype(int).reset_index(drop=True) for y in test_ys]

# drop constant columns based on TRAIN only (correct practice)
X_train = drop_constant_columns(X_train)

# Apply same columns to test blocks
X_test_blocks = [Xt.reindex(columns=X_train.columns) for Xt in X_test_blocks]

print("Train blocks:", train_blocks)
print("Test blocks :", test_blocks)
print("Train shape :", X_train.shape, "pos:", int(y_train.sum()), "pos_rate:", float(y_train.mean()))

# -----------------------------
# Train XGBoost (handles NaN natively)
# -----------------------------
pos = int(y_train.sum())
neg = int(len(y_train) - pos)
scale_pos_weight = (neg / pos) if pos > 0 else 1.0
print("scale_pos_weight:", scale_pos_weight)

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1.0,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",          # PR-AUC is more informative under imbalance
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# -----------------------------
# Evaluate on each test block + overall
# -----------------------------
results = []
all_y = []
all_p = []

for b, Xt, yt in zip(test_blocks, X_test_blocks, y_test_blocks):
    proba = model.predict_proba(Xt)[:, 1]
    results.append(eval_block(yt.values, proba, b))
    all_y.append(yt.values)
    all_p.append(proba)

# overall (concat)
all_y = np.concatenate(all_y)
all_p = np.concatenate(all_p)
results.append(eval_block(all_y, all_p, "ALL_TEST"))

res_df = pd.DataFrame(results)
res_df


Drop constant columns (5): ['driver_is_clkbuf', 'has_multiple_drivers', 'num_clkbuf_drivers', 'num_drivers', 'num_inout']
Train blocks: ['ac97_top', 'i2c_master_top', 'mc_top', 'sasc_top', 'simple_spi_top']
Test blocks : ['spi_top', 'usb_phy']
Train shape : (18585, 61) pos: 913 pos_rate: 0.04912563895614743
scale_pos_weight: 19.355969331872945


,block,n,pos,pos_rate,roc_auc,pr_auc,recall@K(pos),K
0,spi_top,3635,138,0.037964,0.764618,0.107446,0.152174,138
1,usb_phy,526,22,0.041825,0.394210,0.042385,0.045455,22
2,ALL_TEST,4161,160,0.038452,0.720045,0.097879,0.137500,160


In [19]:
df_y = pd.read_csv('../dataset/i2c_master_top/place_label/net_labels.csv', sep=',')

In [6]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_fscore_support

# -----------------------------
# Config
# -----------------------------

blocks = [p.name for p in Path("/home/nzheludkov/phd/ml-critical-path-prediction/dataset").iterdir() if p.is_dir()]

test_blocks = ['FIR_filter', 'i2c_master_top']
train_blocks = [b for b in blocks if b not in test_blocks]

base_dir = "../dataset"
non_feature_cols = ['net_name', 'net_dbid', 'odb_net', 'status', 'label']

# -----------------------------
# Utils
# -----------------------------
def normalize_net_name(s):
    return str(s).strip().replace('"', '')

def load_one_block(block: str) -> pd.DataFrame:
    f_feat = os.path.join(base_dir, block, "features", "net_features_place.csv")
    f_lbl  = os.path.join(base_dir, block, "route_label", "net_labels.csv")

    df = pd.read_csv(f_feat, sep=",")
    df_y = pd.read_csv(f_lbl, sep=",")

    df['net_name'] = df['net_name'].apply(normalize_net_name)
    df_y['net_name'] = df_y['net_name'].apply(normalize_net_name)

    df_all = df.merge(df_y, on='net_name', how='inner')
    df_all['block'] = block
    return df_all

def load_place_labels(block: str) -> pd.DataFrame:
    f_place = os.path.join(base_dir, block, "place_label", "net_labels.csv")
    df_p = pd.read_csv(f_place, sep=",")
    df_p['net_name'] = df_p['net_name'].apply(normalize_net_name)
    # expect columns: net_name,label
    return df_p[['net_name', 'label']].rename(columns={'label': 'place_label'})

def to_numeric_features(df_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    # returns X, y_route, net_names
    net_names = df_all['net_name'].copy()

    x = df_all.drop(columns=[c for c in non_feature_cols if c in df_all.columns], errors='ignore').copy()
    y = df_all['label'].astype(int)

    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors='coerce')

    x = x.dropna(axis=1, how='all')
    return x, y, net_names

def align_feature_columns(X_list: list[pd.DataFrame]) -> tuple[list[pd.DataFrame], list[str]]:
    all_cols = sorted(set().union(*[set(X.columns) for X in X_list]))
    X_aligned = [X.reindex(columns=all_cols) for X in X_list]
    return X_aligned, all_cols

def drop_constant_columns_train_only(X_train: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    nunique = X_train.nunique(dropna=True)
    const_cols = nunique[nunique <= 1].index.tolist()
    if const_cols:
        print(f"Drop constant columns ({len(const_cols)}): {const_cols}")
        X_train = X_train.drop(columns=const_cols)
    else:
        print("Drop constant columns (0)")
    return X_train, const_cols

def recall_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.sum() == 0:
        return np.nan
    k = int(max(1, min(k, len(y_true))))
    idx = np.argsort(y_score)[::-1][:k]
    return float(y_true[idx].sum() / y_true.sum())

def eval_block_auc(y_true, proba, block_name: str, prefix: str):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)

    roc = np.nan
    pr = np.nan
    if len(np.unique(y_true)) > 1:
        roc = roc_auc_score(y_true, proba)
        pr = average_precision_score(y_true, proba)

    k = int(y_true.sum()) if y_true.sum() > 0 else 1
    r_at_k = recall_at_k(y_true, proba, k)

    return {
        "block": block_name,
        "target": prefix,
        "n": int(len(y_true)),
        "pos": int(y_true.sum()),
        "pos_rate": float(y_true.mean()),
        "roc_auc": roc,
        "pr_auc": pr,
        "recall@K(pos)": r_at_k,
        "K": int(k),
    }

def eval_against_place_labels(block: str, net_names: pd.Series, proba: np.ndarray) -> dict:
    """
    Compare model probabilities against place STA labels.
    Uses Top-K where K = number of place positives in the matched intersection.
    Reports: AUCs, recall@K, precision/recall/F1 for Top-K, Jaccard.
    """
    df_place = load_place_labels(block)

    df_pred = pd.DataFrame({"net_name": net_names.values, "p_model": proba})
    df_cmp = df_pred.merge(df_place, on="net_name", how="inner")

    y_true = df_cmp["place_label"].astype(int).values
    p = df_cmp["p_model"].values

    # AUCs
    roc = np.nan
    pr = np.nan
    if len(np.unique(y_true)) > 1:
        roc = roc_auc_score(y_true, p)
        pr = average_precision_score(y_true, p)

    # Top-K budget = #place positives in intersection
    K = int(y_true.sum()) if y_true.sum() > 0 else 1
    idx_topk = np.argsort(p)[::-1][:K]
    y_pred_topk = np.zeros_like(y_true)
    y_pred_topk[idx_topk] = 1

    # Precision/Recall/F1 for Top-K
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_topk, average="binary", zero_division=0
    )

    # Jaccard
    place_set = set(df_cmp.loc[df_cmp["place_label"] == 1, "net_name"])
    pred_set = set(df_cmp.loc[y_pred_topk == 1, "net_name"])
    jacc = len(place_set & pred_set) / max(1, len(place_set | pred_set))

    # Recall@K(pos) relative to place
    r_at_k = recall_at_k(y_true, p, K)

    return {
        "block": block,
        "target": "PLACE_LABEL",
        "n": int(len(y_true)),
        "pos": int(y_true.sum()),
        "pos_rate": float(y_true.mean()) if len(y_true) else np.nan,
        "roc_auc": roc,
        "pr_auc": pr,
        "recall@K(pos)": r_at_k,
        "K": int(K),
        "precision_topK": float(prec),
        "recall_topK": float(rec),
        "f1_topK": float(f1),
        "jaccard_topK": float(jacc),
        "matched_nets": int(len(df_cmp)),
    }

# -----------------------------
# Load data
# -----------------------------
train_dfs = [load_one_block(b) for b in train_blocks]
test_dfs  = [load_one_block(b) for b in test_blocks]

train_Xs, train_ys, _ = zip(*[to_numeric_features(df) for df in train_dfs])
test_Xs,  test_ys,  test_net_names = zip(*[to_numeric_features(df) for df in test_dfs])

# Align columns across all blocks (train+test)
all_Xs = list(train_Xs) + list(test_Xs)
all_Xs_aligned, all_cols = align_feature_columns(all_Xs)

train_Xs_aligned = all_Xs_aligned[:len(train_Xs)]
test_Xs_aligned  = all_Xs_aligned[len(train_Xs):]

# Concatenate train
X_train = pd.concat(train_Xs_aligned, axis=0, ignore_index=True)
y_train = pd.concat([pd.Series(y) for y in train_ys], axis=0, ignore_index=True).astype(int)

# Drop constants on TRAIN only
X_train, const_cols = drop_constant_columns_train_only(X_train)

# Apply same columns to test
X_test_blocks = [Xt.reindex(columns=X_train.columns) for Xt in test_Xs_aligned]
y_test_blocks = [pd.Series(y).astype(int).reset_index(drop=True) for y in test_ys]
net_test_blocks = [s.reset_index(drop=True) for s in test_net_names]

print("Train blocks:", train_blocks)
print("Test blocks :", test_blocks)
print("Train shape :", X_train.shape, "pos:", int(y_train.sum()), "pos_rate:", float(y_train.mean()))

# -----------------------------
# Train XGBoost
# -----------------------------
pos = int(y_train.sum())
neg = int(len(y_train) - pos)
scale_pos_weight = (neg / pos) if pos > 0 else 1.0
print("scale_pos_weight:", scale_pos_weight)

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1.0,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# -----------------------------
# Evaluate on ROUTE labels (per block + overall)
# -----------------------------
results = []

all_y_route = []
all_p = []

for b, Xt, yt in zip(test_blocks, X_test_blocks, y_test_blocks):
    proba = model.predict_proba(Xt)[:, 1]
    results.append(eval_block_auc(yt.values, proba, b, prefix="ROUTE_LABEL"))

    all_y_route.append(yt.values)
    all_p.append(proba)

# overall route
all_y_route = np.concatenate(all_y_route)
all_p_all = np.concatenate(all_p)
results.append(eval_block_auc(all_y_route, all_p_all, "ALL_TEST", prefix="ROUTE_LABEL"))

# -----------------------------
# Evaluate against PLACE labels (per test block + overall)
# -----------------------------
place_results = []
all_place_y = []
all_place_p = []

for b, net_names, proba in zip(test_blocks, net_test_blocks, all_p):
    pr = eval_against_place_labels(b, net_names, proba)
    place_results.append(pr)

    # for overall PLACE_LABEL (intersection per block)
    df_place = load_place_labels(b)
    df_pred = pd.DataFrame({"net_name": net_names.values, "p_model": proba})
    df_cmp = df_pred.merge(df_place, on="net_name", how="inner")
    all_place_y.append(df_cmp["place_label"].astype(int).values)
    all_place_p.append(df_cmp["p_model"].values)

# overall place
if len(all_place_y) > 0:
    all_place_y = np.concatenate(all_place_y)
    all_place_p = np.concatenate(all_place_p)

    # overall AUC
    roc = roc_auc_score(all_place_y, all_place_p) if len(np.unique(all_place_y)) > 1 else np.nan
    pr  = average_precision_score(all_place_y, all_place_p) if len(np.unique(all_place_y)) > 1 else np.nan

    K = int(all_place_y.sum()) if all_place_y.sum() > 0 else 1
    r_at_k = recall_at_k(all_place_y, all_place_p, K)

    # Top-K precision/recall/F1 and Jaccard overall
    idx_topk = np.argsort(all_place_p)[::-1][:K]
    y_pred_topk = np.zeros_like(all_place_y)
    y_pred_topk[idx_topk] = 1
    prec, rec, f1, _ = precision_recall_fscore_support(all_place_y, y_pred_topk, average="binary", zero_division=0)

    # Jaccard overall
    # (use indices because net_name concat differs; jaccard here is still meaningful as set overlap size / union size)
    place_set_idx = set(np.where(all_place_y == 1)[0].tolist())
    pred_set_idx  = set(np.where(y_pred_topk == 1)[0].tolist())
    jacc = len(place_set_idx & pred_set_idx) / max(1, len(place_set_idx | pred_set_idx))

    place_results.append({
        "block": "ALL_TEST",
        "target": "PLACE_LABEL",
        "n": int(len(all_place_y)),
        "pos": int(all_place_y.sum()),
        "pos_rate": float(all_place_y.mean()),
        "roc_auc": roc,
        "pr_auc": pr,
        "recall@K(pos)": r_at_k,
        "K": int(K),
        "precision_topK": float(prec),
        "recall_topK": float(rec),
        "f1_topK": float(f1),
        "jaccard_topK": float(jacc),
        "matched_nets": int(len(all_place_y)),
    })

# -----------------------------
# Show results
# -----------------------------
route_df = pd.DataFrame(results)
place_df = pd.DataFrame(place_results)

print("\n=== ROUTE_LABEL evaluation ===")
display(route_df)

print("\n=== PLACE_LABEL evaluation (model vs STA@place) ===")
display(place_df)

# -----------------------------
# Feature importance (top 30)
# -----------------------------
imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop-30 features by XGBoost importance:")
display(imp.head(30).to_frame("importance"))


Drop constant columns (5): ['driver_is_clkbuf', 'has_multiple_drivers', 'num_clkbuf_drivers', 'num_drivers', 'num_inout']
Train blocks: ['simple_spi_top', 'Md5Core', 'sasc_top', 'usb_phy', 'ac97_top', 'spi_top', 'dynamic_node_top_wrap', 'pcm_slv_top', 'mc_top', 'aes_cipher_top', 'sha256']
Test blocks : ['FIR_filter', 'i2c_master_top']
Train shape : (163082, 61) pos: 10714 pos_rate: 0.06569701131945892
scale_pos_weight: 14.221392570468545

=== ROUTE_LABEL evaluation ===


,block,target,n,pos,pos_rate,roc_auc,pr_auc,recall@K(pos),K
0,FIR_filter,ROUTE_LABEL,3708,285,0.076861,0.797858,0.356487,0.371930,285
1,i2c_master_top,ROUTE_LABEL,1257,19,0.015115,0.773829,0.048975,0.052632,19
2,ALL_TEST,ROUTE_LABEL,4965,304,0.061229,0.809485,0.332587,0.351974,304



=== PLACE_LABEL evaluation (model vs STA@place) ===


,block,target,n,pos,pos_rate,roc_auc,pr_auc,recall@K(pos),K,precision_topK,recall_topK,f1_topK,jaccard_topK,matched_nets
0,FIR_filter,PLACE_LABEL,3708,272,0.073355,0.794627,0.324748,0.363971,272,0.363971,0.363971,0.363971,0.222472,3708
1,i2c_master_top,PLACE_LABEL,1257,17,0.013524,0.767078,0.046038,0.058824,17,0.058824,0.058824,0.058824,0.030303,1257
2,ALL_TEST,PLACE_LABEL,4965,289,0.058207,0.806940,0.303232,0.346021,289,0.346021,0.346021,0.346021,0.209205,4965



Top-30 features by XGBoost importance:


,importance
is_bus_bit,0.318153
driver_is_ff_q,0.112767
driver_is_buf,0.053195
num_buf_drivers,0.052065
has_bterm,0.030645
driver_is_inv,0.026795
term_count,0.023969
num_inv_drivers,0.021067
num_ff_sinks_d,0.018512
sink_area_sum,0.018324
